<a href="https://colab.research.google.com/github/si66326h-cmyk/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/si66326h-cmyk/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*
## 1. My lane as an ML task (type)

**Task Type:** Binary Classification & Priority Ranking

**Why:**
We are solving a **priority ranking** problem supported by a **binary classification model**.
- Content managers have finite weekly hours to update pages. Simply identifying *if* a page is declining is not enough; we must score and rank pages to construct a prioritized work queue.
- The base model predicts the probability that an individual content item (`content_id`) will experience organic traffic decline in a future observation window.
- The predictions are then ordered to produce a top-$K$ priority queue for content strategists.

In [19]:
# Check total dataset size and lane distribution
import pandas as pd

url = "https://raw.githubusercontent.com/si66326h-cmyk/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# Compute target label from observed trend_direction
df["is_declining_label"] = (df["trend_direction"] == "declining").astype(int)

print(f"Total Rows: {len(df)}")
print(f"Unique Clients: {df['client_id'].nunique()}")
print(f"Unique Content Items: {df['content_id'].nunique()}")
print("\nContent Type Distribution:")
print(df["content_type"].value_counts(dropna=False))

Total Rows: 30000
Unique Clients: 32
Unique Content Items: 30000

Content Type Distribution:
content_type
keyword article       27207
feedly article         2096
comparison article      697
Name: count, dtype: int64


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*
## 2. Target or proxy

**Target Label:** `is_declining_label` (Binary: `1` for declining, `0` for non-declining).

**Source & Integrity Rules:**
- **Observed Outcome:** The target is an observed future outcome derived from traffic performance across evaluation windows, not a heuristic rule defined at inference time.
- **Leakage Prevention:** Features `trend_pct` and `trend_direction` must be **strictly removed** from the feature matrix $X$. `is_declining_label` is derived directly from `trend_pct`, making both direct target leakage.
- **Split Strategy:** Validation splits must use `GroupKFold(groups=df['client_id'])` so that all records for a given client remain in either the train or validation set, preventing domain-level leakage.

In [20]:
# Verify target label presence and confirm leakage columns to drop
target_str = "is_declining_label"
leakage_cols = ["trend_pct", "trend_direction"]

print("Target Class Distribution:")
print(df[target_str].value_counts(normalize=True).rename({1: "Declining (1)", 0: "Healthy (0)"}))

# Demonstrate feature isolation
exclude_cols = leakage_cols + [target_str]
feature_candidates = [c for c in df.columns if c not in exclude_cols]
print(f"\nTotal Features (excluding target & leakage): {len(feature_candidates)}")

Target Class Distribution:
is_declining_label
Healthy (0)    1.0
Name: proportion, dtype: float64

Total Features (excluding target & leakage): 42


## 3. Success metric

*One metric you can defend. What number means 'good'?*
## 3. Success metric

**Primary Metric:** **Precision@K** (e.g., $K = 50$ top-ranked pages per client)

**Defense:**
- **Decision Alignment:** Content strategists can only rewrite a fixed number of pages per month ($K$). Maximizing Precision@K ensures that when a strategist picks up an item from the top of the queue, it is genuinely declining and worth rewriting.
- **False Positive Cost:** High false positive rates waste high-value editorial/SEO bandwidth on healthy pages that do not require updates.
- **Secondary Metric:** **Grouped ROC-AUC (by client)** to verify that ranking performance remains consistent across clients with varying baseline traffic levels.

In [21]:
# Baseline check of target rate across clients to justify Grouped Precision@K / ROC-AUC
target_str = "is_declining_label"

client_stats = (
    df.groupby("client_id")[target_str]
    .agg(total_items="count", decline_rate="mean")
    .reset_index()
)

print("Decline rate summary across clients:")
print(client_stats["decline_rate"].describe())

Decline rate summary across clients:
count    32.0
mean      0.0
std       0.0
min       0.0
25%       0.0
50%       0.0
75%       0.0
max       0.0
Name: decline_rate, dtype: float64


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Unit of Analysis:** One row = **One Content Item (`content_id`) for a specific Client (`client_id`)**.

In [22]:
import numpy as np

# 1. Clean scale/encoding anomalies (avg_position = 0 means 'no rank/data', not position 0)
df["avg_position_clean"] = df["avg_position"].replace(0, np.nan)

# 2. Add binary missingness indicator flags for features tied to content_type
df["has_scroll_rate"] = df["scroll_rate"].notna().astype(int)
df["has_engagement_rate"] = df["engagement_rate"].notna().astype(int)

# 3. Define schema display columns
id_cols = ["content_id", "client_id", "content_type"]
metric_cols = [
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position_clean",
    "has_scroll_rate",
    "has_engagement_rate",
]
target_col_list = ["is_declining_label"]

# Inspect unit of analysis schema
unit_slice = df[id_cols + metric_cols + target_col_list].head(5)
print(f"Unit of analysis shape: {unit_slice.shape}")
unit_slice

Unit of analysis shape: (5, 10)


,content_id,client_id,content_type,impressions_90d,clicks_90d,ctr,avg_position_clean,has_scroll_rate,has_engagement_rate,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,3803,29,0.76,10.6,1,1,0
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,7,0.05,20.3,1,1,0
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,11,0.09,36.5,1,1,0
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,58,0.49,6.2,1,1,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,24,0.13,44.0,1,1,0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*


**Why Static Rules Fail:**
1. **Domain Heterogeneity:** A rule like `"Clicks dropped > 20%"` breaks across clients. Large media sites experience normal seasonal fluctuations of 20%, whereas B2B sites with low baseline volume may drop 50% from a single missing conversion.
2. **Multi-Signal Interactions:** Decay is non-linear. A page might maintain high click volume while its average search rank (`avg_position`) steadily degrades and user scroll depth (`scroll_rate`) drops—signaling impending decay before total traffic falls off.
3. **Missingness Signals:** Data metrics vary by content type (e.g., non-article pages naturally lack scroll depth). ML models with explicit missingness flags (`has_<col>`) learn patterns from available signals without introducing false noise from naive zero-imputation.

In [23]:
# Show non-linear variation: Click volume vs. decline state across different content types
target_str = "is_declining_label"

analysis = df.groupby(["content_type", target_str])[["clicks_90d", "ctr", "avg_position_clean"]].mean()
print("Average metrics by Content Type and Decline State:")
print(analysis)

Average metrics by Content Type and Decline State:
                                       clicks_90d       ctr  \
content_type       is_declining_label                         
comparison article 0                     0.255380  0.131205   
feedly article     0                     1.072996  2.791274   
keyword article    0                    17.660639  0.344766   

                                       avg_position_clean  
content_type       is_declining_label                      
comparison article 0                            11.296844  
feedly article     0                             8.856223  
keyword article    0                            17.593143  


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.